In [ ]:
import json
import os
import time
import warnings
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from pennylane import numpy as pnp

warnings.filterwarnings("ignore")

# =============================================================================
# User Config
# =============================================================================
N_QUBITS = 4
N_LAYERS = 2
WIRES = list(range(N_QUBITS))
RANGES = [1, 2]   # len == N_LAYERS

# -----------------------------------------------------------------------------
# Optimizer choices for each method: "gd", "adam", or "spsa"
# -----------------------------------------------------------------------------
# Paper-style comparison이면 clean / shot은 gd가 맞음
CLEAN_OPTIMIZER = "adam"
SHOT_OPTIMIZER = "adam"
PAULI_OPTIMIZER = "adam"

# Separate learning rates for GD / Adam
LR_CLEAN_GD = 0.1
LR_CLEAN_ADAM = 0.01

LR_SHOT_GD = 0.1
LR_SHOT_ADAM = 0.01

LR_PAULI_GD = 0.1
LR_PAULI_ADAM = 0.01

# -----------------------------------------------------------------------------
# Separate SPSA hyperparameters for each method
# qml.SPSAOptimizer(maxiter, alpha, gamma, c, A, a)
# -----------------------------------------------------------------------------
# 너무 출렁이고 불안정하면: a를 줄여라, 거의 안 움직이면: a를 키워라
# gradient estimate가 너무 noisy해 보이면: c를 조금 키워라, 너무 거칠게 흔들리면: c를 줄여라
CLEAN_SPSA_ALPHA = 1 # 0.602 # optimal 1
CLEAN_SPSA_GAMMA = 1/6 #0.101 # optimal 1/6
CLEAN_SPSA_C = 0.2 # default 0.2
CLEAN_SPSA_A = 0
CLEAN_SPSA_a = 3

SHOT_SPSA_ALPHA = 1
SHOT_SPSA_GAMMA = 1/6
SHOT_SPSA_C = 0.2 #0.10
SHOT_SPSA_A = 0 #None
SHOT_SPSA_a = 3 #0.05

PAULI_SPSA_ALPHA = 1
PAULI_SPSA_GAMMA = 1/6
PAULI_SPSA_C = 0.2 #0.10
PAULI_SPSA_A = 0 #None
PAULI_SPSA_a = 3 #0.05

# -----------------------------------------------------------------------------
# Seed control
# -----------------------------------------------------------------------------
# False면 INIT_SEED를 그대로 사용하고 search를 생략
RUN_SEED_SEARCH = True
INIT_SEED = 1

# Search config (used only if RUN_SEED_SEARCH=True)
# 논문식 trapped count를 보려면 1000개 seed를 확인
SEARCH_SEEDS = list(range(1000))
SEARCH_STEPS = 500

# Hessian-based diagnosis용
SUBOPT_GAP = 0.25
GRAD_TOL = 1e-4
HESS_NEG_TOL = 1e-5

# Paper-style trapped 판정용
# final gap이 충분히 크고 마지막 구간이 평평하면 trapped로 간주
PAPER_TAIL_WINDOW = 20
PAPER_TAIL_STD_TOL = 1e-2
PAPER_TAIL_DRIFT_TOL = 1e-2

# trapped stationary 후보 중 대표 seed 선택 score
TRAP_SCORE_W_STD = 1.0
TRAP_SCORE_W_DRIFT = 1.0
TRAP_SCORE_W_GRAD = 0.5
TRAP_SCORE_W_GAP = 0.05   # subtract됨: gap이 큰 trapped seed를 약간 선호

# -----------------------------------------------------------------------------
# Adaptive Pauli annealing schedule
# -----------------------------------------------------------------------------
PAULI_SCHEDULE = [0.4, 0.3, 0.2, 0.1, 0.05, 0.03, 0.01, 0.007, 0.005, 0.0]

PAULI_MIN_STEPS = 50
PAULI_MAX_STEPS = 1000
PAULI_CHECK_EVERY = 10
PAULI_WINDOW = 20
PAULI_STD_TOL = 0.005
PAULI_RATE_TOL = 0.005
CLEAN_THRESHOLD = 0.005

# -----------------------------------------------------------------------------
# Shot-noise config
# -----------------------------------------------------------------------------
# 논문 비교용 대표 shots
SHOT_LIST = [1000, 500, 100, 50]
SHOT_REPEATS = 10
SHOT_DEVICE_BASE_SEED = 12345

# Escape criterion for shot-noise
# None이면 clean baseline final energy - SHOT_ESCAPE_DELTA
SHOT_ESCAPE_THRESHOLD = None
SHOT_ESCAPE_DELTA = 0.5

# -----------------------------------------------------------------------------
# Save
# -----------------------------------------------------------------------------
SAVE_PREFIX = "adaptive_pauli_vs_shot_sel"

# Exact ground energy for H = sum_i Z_i
GLOBAL_MIN_ENERGY = -float(N_QUBITS)


# =============================================================================
# Hamiltonian
# =============================================================================
def build_local_z_hamiltonian(n_qubits: int):
    coeffs = [1.0] * n_qubits
    obs = [qml.PauliZ(i) for i in range(n_qubits)]
    return qml.Hamiltonian(coeffs, obs)


H = build_local_z_hamiltonian(N_QUBITS)


# =============================================================================
# Ansatz application
# =============================================================================
def apply_sel(weights):
    qml.StronglyEntanglingLayers(
        weights,
        wires=WIRES,
        ranges=RANGES,
        imprimitive=qml.CNOT,
    )


def apply_sel_with_layerwise_pauli(weights, noise_p: float):
    """
    Layerwise Pauli insertion:
    before each StronglyEntanglingLayers layer, insert X/Z/Y PauliError
    on every qubit, then apply that one SEL layer.
    """
    p = np.clip(float(noise_p), 1e-12, 1.0 - 1e-12)

    for l in range(N_LAYERS):
        for w in WIRES:
            qml.PauliError("X", p, wires=w)
            qml.PauliError("Z", p, wires=w)
            qml.PauliError("Y", p, wires=w)

        qml.StronglyEntanglingLayers(
            weights[l:l+1],
            wires=WIRES,
            ranges=[RANGES[l]],
            imprimitive=qml.CNOT,
        )


# =============================================================================
# Cost function factories
# =============================================================================
def make_clean_cost():
    dev = qml.device("default.qubit", wires=N_QUBITS)

    @qml.qnode(dev, interface="autograd", diff_method="best")
    def cost(weights):
        apply_sel(weights)
        return qml.expval(H)

    return cost


def make_pauli_cost(noise_p: float):
    dev = qml.device("default.mixed", wires=N_QUBITS)

    @qml.qnode(dev, interface="autograd", diff_method="best")
    def cost(weights):
        apply_sel_with_layerwise_pauli(weights, noise_p)
        return qml.expval(H)

    return cost


def make_shot_cost(shots: int, device_seed: int):
    dev = qml.device(
        "default.qubit",
        wires=N_QUBITS,
        shots=shots,
        seed=device_seed,
    )

    @qml.qnode(dev, interface="autograd", diff_method="best")
    def cost(weights):
        apply_sel(weights)
        return qml.expval(H)

    return cost


CLEAN_COST = make_clean_cost()


# =============================================================================
# Init
# =============================================================================
def init_params(seed: int):
    rng = np.random.default_rng(seed)
    arr = rng.uniform(0.0, 2.0 * np.pi, size=(N_LAYERS, N_QUBITS, 3))
    return pnp.array(arr, requires_grad=True)


# =============================================================================
# Diagnostics on clean landscape
# =============================================================================
def clean_hessian_info(params):
    shape = (N_LAYERS, N_QUBITS, 3)
    x0 = pnp.array(np.array(params).reshape(-1), requires_grad=True)

    def flat_clean_cost(x):
        w = x.reshape(shape)
        return CLEAN_COST(w)

    grad_fn = qml.grad(flat_clean_cost)
    hess_fn = qml.jacobian(grad_fn)

    g = np.array(grad_fn(x0), dtype=float)
    Hh = np.array(hess_fn(x0), dtype=float)
    Hh = 0.5 * (Hh + Hh.T)
    eigvals = np.linalg.eigvalsh(Hh)

    return {
        "grad_norm": float(np.linalg.norm(g)),
        "lambda_min": float(eigvals[0]),
        "lambda_max": float(eigvals[-1]),
    }


# =============================================================================
# Generic optimizer helpers
# =============================================================================
def validate_optimizer_name(name: str):
    name = name.lower()
    if name not in {"gd", "adam", "spsa"}:
        raise ValueError(f"optimizer must be 'gd', 'adam', or 'spsa', got {name}")
    return name


def _to_trainable_params(x):
    return pnp.array(np.array(x), requires_grad=True)


def _unwrap_single_arg_step_output(x):
    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            raise ValueError("optimizer step returned an empty container")
        return x[0]
    return x


def make_stepper(
    cost_fn,
    optimizer_type="gd",
    max_steps_for_spsa=100,
    lr_gd=0.2,
    lr_adam=0.01,
    spsa_alpha=0.602,
    spsa_gamma=0.101,
    spsa_c=0.2,
    spsa_A=None,
    spsa_a=None,
):
    optimizer_type = validate_optimizer_name(optimizer_type)

    if optimizer_type == "adam":
        opt = qml.AdamOptimizer(stepsize=lr_adam)

        def one_step(cur_params):
            new_params = opt.step(cost_fn, cur_params)
            return _to_trainable_params(_unwrap_single_arg_step_output(new_params))

        opt_desc = f"adam(lr={lr_adam})"

    elif optimizer_type == "gd":
        grad_fn = qml.grad(cost_fn)

        def one_step(cur_params):
            grads = grad_fn(cur_params)
            return _to_trainable_params(cur_params - lr_gd * grads)

        opt_desc = f"gd(lr={lr_gd})"

    else:
        opt = qml.SPSAOptimizer(
            maxiter=max(1, int(max_steps_for_spsa)),
            alpha=spsa_alpha,
            gamma=spsa_gamma,
            c=spsa_c,
            A=spsa_A,
            a=spsa_a,
        )

        def one_step(cur_params):
            new_params = opt.step(cost_fn, cur_params)
            return _to_trainable_params(_unwrap_single_arg_step_output(new_params))

        opt_desc = (
            "spsa("
            f"alpha={spsa_alpha}, gamma={spsa_gamma}, c={spsa_c}, "
            f"A={spsa_A}, a={spsa_a}, maxiter={max(1, int(max_steps_for_spsa))}"
            ")"
        )

    return one_step, opt_desc


def optimize_fixed_steps(
    cost_fn,
    params0,
    steps,
    optimizer_type="gd",
    lr_gd=0.2,
    lr_adam=0.01,
    spsa_alpha=0.602,
    spsa_gamma=0.101,
    spsa_c=0.2,
    spsa_A=None,
    spsa_a=None,
    eval_fn=None,
):
    """
    Generic optimizer runner.

    train_hist: objective actually optimized
    eval_hist : clean exact re-evaluation at every step
    """
    optimizer_type = validate_optimizer_name(optimizer_type)

    if eval_fn is None:
        eval_fn = cost_fn

    params = _to_trainable_params(params0)

    one_step, opt_desc = make_stepper(
        cost_fn=cost_fn,
        optimizer_type=optimizer_type,
        max_steps_for_spsa=steps,
        lr_gd=lr_gd,
        lr_adam=lr_adam,
        spsa_alpha=spsa_alpha,
        spsa_gamma=spsa_gamma,
        spsa_c=spsa_c,
        spsa_A=spsa_A,
        spsa_a=spsa_a,
    )

    train_hist = [float(cost_fn(params))]
    eval_hist = [float(eval_fn(params))]

    for _ in range(steps):
        params = one_step(params)
        train_hist.append(float(cost_fn(params)))
        eval_hist.append(float(eval_fn(params)))

    return {
        "final_params": _to_trainable_params(params),
        "train_hist": np.array(train_hist),
        "eval_hist": np.array(eval_hist),
        "final_train": float(train_hist[-1]),
        "final_eval": float(eval_hist[-1]),
        "total_steps": int(steps),
        "optimizer_type": optimizer_type,
        "optimizer_desc": opt_desc,
        "lr_gd": float(lr_gd),
        "lr_adam": float(lr_adam),
        "spsa_alpha": float(spsa_alpha),
        "spsa_gamma": float(spsa_gamma),
        "spsa_c": float(spsa_c),
        "spsa_A": None if spsa_A is None else float(spsa_A),
        "spsa_a": None if spsa_a is None else float(spsa_a),
    }


# =============================================================================
# Seed diagnosis/search
# =============================================================================
def classify_seed_from_clean_result(clean_result, hessian_info):
    final_e = float(clean_result["final_eval"])
    gap = final_e - GLOBAL_MIN_ENERGY

    is_stationary = hessian_info["grad_norm"] < GRAD_TOL
    is_strict_saddle_like = is_stationary and (hessian_info["lambda_min"] < -HESS_NEG_TOL)

    if gap <= SUBOPT_GAP:
        kind = "not-suboptimal-or-too-close-to-ground"
    elif is_strict_saddle_like:
        kind = "strict-saddle-like"
    elif is_stationary:
        kind = "stationary-suboptimal"
    else:
        kind = "suboptimal-but-nonstationary"

    return kind, float(gap)


def paper_trap_info(
    clean_result,
    tail_window=PAPER_TAIL_WINDOW,
    tail_std_tol=PAPER_TAIL_STD_TOL,
    tail_drift_tol=PAPER_TAIL_DRIFT_TOL,
):
    """
    논문식 trapped 판정:
    - final energy가 ground보다 충분히 위에 있고
    - 마지막 구간이 거의 평평하면 trapped로 본다.
    """
    hist = np.array(clean_result["eval_hist"], dtype=float)
    final_e = float(clean_result["final_eval"])
    gap = final_e - GLOBAL_MIN_ENERGY

    w = min(tail_window, len(hist))
    tail = hist[-w:]
    tail_std = float(np.std(tail))
    tail_drift = float(abs(tail[-1] - tail[0]))

    trapped = (gap > SUBOPT_GAP) and (tail_std < tail_std_tol) and (tail_drift < tail_drift_tol)

    return {
        "trapped": bool(trapped),
        "tail_std": tail_std,
        "tail_drift": tail_drift,
        "gap": float(gap),
    }


def stationary_trap_score(clean_result, hessian_info):
    """
    stationary-suboptimal 후보들 중에서
    '가장 전형적으로 trapped된' seed를 고르기 위한 score.
    작을수록 더 전형적 trapped seed로 봄.
    """
    hist = np.array(clean_result["eval_hist"], dtype=float)
    w = min(PAPER_TAIL_WINDOW, len(hist))
    tail = hist[-w:]

    flatness_std = float(np.std(tail))
    flatness_drift = float(abs(tail[-1] - tail[0]))
    grad_norm = float(hessian_info["grad_norm"])

    final_e = float(clean_result["final_eval"])
    gap = final_e - GLOBAL_MIN_ENERGY

    score = (
        TRAP_SCORE_W_STD * flatness_std
        + TRAP_SCORE_W_DRIFT * flatness_drift
        + TRAP_SCORE_W_GRAD * grad_norm
        - TRAP_SCORE_W_GAP * gap
    )

    return {
        "score": float(score),
        "flatness_std": flatness_std,
        "flatness_drift": flatness_drift,
        "gap": float(gap),
    }


def diagnose_seed(seed: int):
    """
    Diagnose any given seed using the CLEAN optimizer setting.
    """
    p0 = init_params(seed)

    res = optimize_fixed_steps(
        cost_fn=CLEAN_COST,
        params0=p0,
        steps=SEARCH_STEPS,
        optimizer_type=CLEAN_OPTIMIZER,
        lr_gd=LR_CLEAN_GD,
        lr_adam=LR_CLEAN_ADAM,
        spsa_alpha=CLEAN_SPSA_ALPHA,
        spsa_gamma=CLEAN_SPSA_GAMMA,
        spsa_c=CLEAN_SPSA_C,
        spsa_A=CLEAN_SPSA_A,
        spsa_a=CLEAN_SPSA_a,
        eval_fn=CLEAN_COST,
    )
    info = clean_hessian_info(res["final_params"])
    kind, gap = classify_seed_from_clean_result(res, info)
    trap_info = paper_trap_info(res)
    trap_score_info = stationary_trap_score(res, info)

    return {
        "seed": int(seed),
        "init_params": p0,
        "clean_result": res,
        "hessian_info": info,
        "kind": kind,
        "gap": gap,
        "paper_trap_info": trap_info,
        "trap_score_info": trap_score_info,
    }


def is_problematic_seed(target):
    """
    실제 선택 후보군에 들어가는 seed 정의:
    - strict-saddle-like
    - stationary-suboptimal
    - 또는 paper-trapped
    """
    kind = target["kind"]
    return (kind in {"strict-saddle-like", "stationary-suboptimal"}) or bool(target["paper_trap_info"]["trapped"])


def paper_only_score(target):
    """
    paper-trapped fallback에서 실제 사용하는 점수.
    """
    return (
        float(target["paper_trap_info"]["tail_std"])
        + float(target["paper_trap_info"]["tail_drift"])
        - 0.05 * float(target["gap"])
    )


def actual_selection_sort_key(target):
    """
    현재 find_saddle_like_seed()의 실제 선택 로직과 동일한 정렬 key.

    우선순위:
    1) strict-saddle-like -> seed가 작은 순
    2) stationary-suboptimal -> trap_score가 작은 순
    3) paper-trapped-only -> paper_only_score가 작은 순
    4) 나머지 -> 맨 뒤
    """
    kind = target["kind"]
    seed = int(target["seed"])

    if kind == "strict-saddle-like":
        return (0, seed)

    if kind == "stationary-suboptimal":
        return (1, float(target["trap_score_info"]["score"]), seed)

    if bool(target["paper_trap_info"]["trapped"]):
        return (2, paper_only_score(target), seed)

    return (3, seed)


def sort_problematic_seeds(problematic_list):
    """
    현재 실제 대표 seed 선택 기준과 동일하게 정렬.
    """
    return sorted(problematic_list, key=actual_selection_sort_key)


def print_problematic_seeds(problematic_sorted):
    """
    problematic seed 전체를 화면에 정렬된 표 형태로 출력.
    정렬 기준은 현재 실제 선택 로직과 동일하다.
    """
    print("\n" + "=" * 195)
    print("Problematic seeds sorted by CURRENT ACTUAL selection rule")
    print("priority: strict-saddle-like (smallest seed) -> stationary-suboptimal (min trap_score) -> paper-trapped (min tail_std + tail_drift - 0.05 * gap)")
    print("=" * 195)
    print(
        f"{'rank':>4} | {'seed':>4} | {'kind':>26} | {'paper':>5} | "
        f"{'trap_score':>11} | {'paper_score':>11} | {'gap':>10} | "
        f"{'tail_std':>10} | {'tail_drift':>10} | {'grad_norm':>11} | {'lambda_min':>11}"
    )
    print("-" * 195)

    for rank, item in enumerate(problematic_sorted, start=1):
        print(
            f"{rank:>4} | "
            f"{item['seed']:>4} | "
            f"{item['kind']:>26} | "
            f"{str(item['paper_trap_info']['trapped']):>5} | "
            f"{item['trap_score_info']['score']:>11.3e} | "
            f"{paper_only_score(item):>11.3e} | "
            f"{item['gap']:>10.6f} | "
            f"{item['paper_trap_info']['tail_std']:>10.3e} | "
            f"{item['paper_trap_info']['tail_drift']:>10.3e} | "
            f"{item['hessian_info']['grad_norm']:>11.3e} | "
            f"{item['hessian_info']['lambda_min']:>11.3e}"
        )

    print("=" * 195)


def find_saddle_like_seed():
    """
    Paper-style trapped seed search:
    - 1000개 초기값을 동일한 dynamics로 돌려
    - noiseless case에서 suboptimal plateau에 갇힌 seed를 찾는다.

    선택 로직:
    1) strict-saddle-like가 있으면 그중 첫 번째
    2) 없으면 stationary-suboptimal 중 trap_score 최소
    3) 그것도 없으면 paper-trapped 중 paper_only_score 최소

    출력 정렬도 위 실제 선택 로직과 동일하게 맞춘다.
    """
    strict_saddle_candidates = []
    stationary_fallback = []
    paper_trapped_candidates = []
    all_problematic = []

    print("=" * 80)
    print("Searching for paper-style trapped initializations ...")
    print(f"Search seeds : {SEARCH_SEEDS[0]} ... {SEARCH_SEEDS[-1]}")
    print(f"Search steps : {SEARCH_STEPS}")
    print(f"Clean optimizer for search : {CLEAN_OPTIMIZER}")
    print("=" * 80)

    t0 = time.time()

    for idx, seed in enumerate(SEARCH_SEEDS, start=1):
        target = diagnose_seed(seed)
        kind = target["kind"]

        if kind == "strict-saddle-like":
            strict_saddle_candidates.append(target)

        if kind == "stationary-suboptimal":
            stationary_fallback.append(target)

        if target["paper_trap_info"]["trapped"]:
            paper_trapped_candidates.append(target)

        if is_problematic_seed(target):
            all_problematic.append(target)

        if idx % 50 == 0:
            print(f"  checked {idx:4d}/{len(SEARCH_SEEDS)} seeds ...")

    n_total = len(SEARCH_SEEDS)
    n_paper_trapped = len(paper_trapped_candidates)
    n_strict = len(strict_saddle_candidates)
    n_stationary = len(stationary_fallback)
    n_problematic = len(all_problematic)

    print("\n" + "-" * 80)
    print(f"All problematic count        = {n_problematic}/{n_total} ({100.0*n_problematic/n_total:.2f}%)")
    print(f"Paper-style trapped count    = {n_paper_trapped}/{n_total} ({100.0*n_paper_trapped/n_total:.2f}%)")
    print(f"strict-saddle-like count     = {n_strict}/{n_total}")
    print(f"stationary-suboptimal count  = {n_stationary}/{n_total}")

    problematic_sorted = sort_problematic_seeds(all_problematic)
    if len(problematic_sorted) > 0:
        print_problematic_seeds(problematic_sorted)

    if n_strict > 0:
        best = strict_saddle_candidates[0]
        print("\nUsing strict-saddle-like seed:")
        print(
            f"seed={best['seed']} | "
            f"E={best['clean_result']['final_eval']:.8f} | "
            f"gap={best['gap']:.8f} | "
            f"||grad||={best['hessian_info']['grad_norm']:.3e} | "
            f"lambda_min={best['hessian_info']['lambda_min']:.3e}"
        )
        print(f"Search time: {time.time()-t0:.1f}s")
        return {
            "selected": best,
            "problematic_sorted": problematic_sorted,
        }

    if n_stationary > 0:
        best = min(
            stationary_fallback,
            key=lambda x: x["trap_score_info"]["score"]
        )
        print("\n[WARNING] No strict-saddle-like point found.")
        print("Using most typically trapped stationary-suboptimal point:")
        print(
            f"seed={best['seed']} | "
            f"E={best['clean_result']['final_eval']:.8f} | "
            f"gap={best['gap']:.8f} | "
            f"||grad||={best['hessian_info']['grad_norm']:.3e} | "
            f"lambda_min={best['hessian_info']['lambda_min']:.3e} | "
            f"tail_std={best['trap_score_info']['flatness_std']:.3e} | "
            f"tail_drift={best['trap_score_info']['flatness_drift']:.3e} | "
            f"score={best['trap_score_info']['score']:.3e}"
        )
        print(f"Search time: {time.time()-t0:.1f}s")
        return {
            "selected": best,
            "problematic_sorted": problematic_sorted,
        }

    if n_paper_trapped > 0:
        best = min(
            paper_trapped_candidates,
            key=paper_only_score,
        )
        print("\n[WARNING] No strict-saddle-like or stationary-suboptimal point found.")
        print("Using representative paper-style trapped seed:")
        print(
            f"seed={best['seed']} | "
            f"E={best['clean_result']['final_eval']:.8f} | "
            f"gap={best['gap']:.8f} | "
            f"tail_std={best['paper_trap_info']['tail_std']:.3e} | "
            f"tail_drift={best['paper_trap_info']['tail_drift']:.3e} | "
            f"paper_score={paper_only_score(best):.3e}"
        )
        print(f"Search time: {time.time()-t0:.1f}s")
        return {
            "selected": best,
            "problematic_sorted": problematic_sorted,
        }

    raise RuntimeError(
        "No paper-style trapped seed found. "
        "Try matching the paper dynamics more closely."
    )


# =============================================================================
# Convergence checker
# =============================================================================
def _converged_ckpt(history_ckpt, win_ckpt, std_tol, rate_tol):
    if len(history_ckpt) < 2 * win_ckpt:
        return False, float("inf"), float("inf")

    recent = np.array(history_ckpt[-win_ckpt:], dtype=float)
    prev = np.array(history_ckpt[-2 * win_ckpt:-win_ckpt], dtype=float)

    std = float(np.std(recent))
    delta = float(abs(np.mean(recent) - np.mean(prev)))
    return (std < std_tol and delta < rate_tol), std, delta


# =============================================================================
# Adaptive Pauli annealing
# =============================================================================
def run_pauli_annealing(
    init_params_same,
    noise_schedule,
    optimizer_type=PAULI_OPTIMIZER,
    lr_gd=LR_PAULI_GD,
    lr_adam=LR_PAULI_ADAM,
    spsa_alpha=PAULI_SPSA_ALPHA,
    spsa_gamma=PAULI_SPSA_GAMMA,
    spsa_c=PAULI_SPSA_C,
    spsa_A=PAULI_SPSA_A,
    spsa_a=PAULI_SPSA_a,
    min_steps=PAULI_MIN_STEPS,
    max_steps=PAULI_MAX_STEPS,
    check_every=PAULI_CHECK_EVERY,
    window_steps=PAULI_WINDOW,
    std_tol=PAULI_STD_TOL,
    rate_tol=PAULI_RATE_TOL,
):
    optimizer_type = validate_optimizer_name(optimizer_type)

    params = _to_trainable_params(init_params_same)

    win_ckpt = int(np.ceil(window_steps / check_every))
    win_ckpt = max(2, win_ckpt)

    full_hist_ckpt = []
    full_hist_steps = []

    stage_hists = []
    stage_hist_steps = []
    stage_steps = []
    stage_stds = []
    stage_converged = []
    stage_last_train = []

    total_steps = 0

    for stage_idx, p in enumerate(noise_schedule, start=1):
        use_clean = (p <= CLEAN_THRESHOLD)
        cost_fn = CLEAN_COST if use_clean else make_pauli_cost(p)

        stage_step, opt_desc = make_stepper(
            cost_fn=cost_fn,
            optimizer_type=optimizer_type,
            max_steps_for_spsa=max_steps,
            lr_gd=lr_gd,
            lr_adam=lr_adam,
            spsa_alpha=spsa_alpha,
            spsa_gamma=spsa_gamma,
            spsa_c=spsa_c,
            spsa_A=spsa_A,
            spsa_a=spsa_a,
        )

        stage_hist = []
        stage_hist_x = []

        step_i = 0
        last_clean_e = float(CLEAN_COST(params))

        print(
            f"\n[Pauli Stage {stage_idx}/{len(noise_schedule)}] "
            f"p={p:.3f} ({'clean' if use_clean else 'noisy'}) | opt={opt_desc}"
        )
        print(f"  {'step':>5} | {'E_clean':>12} | {'std':>9} | {'|Δmean|':>9} | note")

        t0 = time.time()
        while step_i < max_steps:
            params = stage_step(params)
            step_i += 1

            if step_i % check_every == 0:
                last_clean_e = float(CLEAN_COST(params))
                stage_hist.append(last_clean_e)
                stage_hist_x.append(total_steps + step_i)

                conv, std, delta = _converged_ckpt(stage_hist, win_ckpt, std_tol, rate_tol)
                note = "<-- converged" if (step_i >= min_steps and conv) else ""
                print(f"  {step_i:>5} | {last_clean_e:>12.6f} | {std:>9.5f} | {delta:>9.5f} | {note}")

                if step_i >= min_steps and conv:
                    break

        conv, std, delta = _converged_ckpt(stage_hist, win_ckpt, std_tol, rate_tol)
        last_train = float(cost_fn(params))

        if (step_i >= max_steps) and (not conv):
            print(f"  *** max_steps={max_steps} reached without convergence ***")

        print(f"  Stage done: steps={step_i}, std={std:.5f}, |Δmean|={delta:.5f}, "
              f"time={time.time()-t0:.1f}s")

        stage_hists.append(np.array(stage_hist, dtype=float))
        stage_hist_steps.append(np.array(stage_hist_x, dtype=int))
        stage_steps.append(int(step_i))
        stage_stds.append(float(std))
        stage_converged.append(bool(conv))
        stage_last_train.append(float(last_train))

        full_hist_ckpt.extend(stage_hist)
        full_hist_steps.extend(stage_hist_x)
        total_steps += step_i

    final_clean_eval = float(CLEAN_COST(params))
    final_hess = clean_hessian_info(params)

    return {
        "history": np.array(full_hist_ckpt, dtype=float),
        "history_steps": np.array(full_hist_steps, dtype=int),
        "stage_hists": stage_hists,
        "stage_hist_steps": stage_hist_steps,
        "stage_steps": stage_steps,
        "stage_stds": stage_stds,
        "stage_converged": stage_converged,
        "stage_last_train": stage_last_train,
        "noise_schedule": list(noise_schedule),
        "optimizer_type": optimizer_type,
        "lr_gd": float(lr_gd),
        "lr_adam": float(lr_adam),
        "spsa_alpha": float(spsa_alpha),
        "spsa_gamma": float(spsa_gamma),
        "spsa_c": float(spsa_c),
        "spsa_A": None if spsa_A is None else float(spsa_A),
        "spsa_a": None if spsa_a is None else float(spsa_a),
        "final_params": _to_trainable_params(params),
        "final_clean_eval": float(final_clean_eval),
        "final_grad_norm_clean_landscape": float(final_hess["grad_norm"]),
        "final_lambda_min_clean_landscape": float(final_hess["lambda_min"]),
        "win_ckpt": int(win_ckpt),
        "total_steps": int(total_steps),
    }


# =============================================================================
# Shot-noise summary helpers
# =============================================================================
def summarize_shot_runs_for_plot(repeats, clean_ref_final, escape_threshold):
    rep_summaries = []

    for rep in repeats:
        hist = np.array(rep["eval_hist"], dtype=float)
        final_e = float(hist[-1])
        min_e = float(np.min(hist))
        escaped = bool(min_e <= escape_threshold)

        rep_summaries.append({
            "rep": rep["rep"],
            "eval_hist": hist,
            "final_clean_eval": final_e,
            "min_clean_eval_over_traj": min_e,
            "escaped": escaped,
        })

    best_idx = int(np.argmin([r["min_clean_eval_over_traj"] for r in rep_summaries]))
    best_rep = rep_summaries[best_idx]

    escape_count = int(sum(r["escaped"] for r in rep_summaries))
    num_repeats = len(rep_summaries)

    if escape_count == 0:
        plot_mode = "single_stuck_only"
        single_rep = best_rep
        stuck_rep = None
    elif escape_count == num_repeats:
        plot_mode = "single_best_only"
        single_rep = best_rep
        stuck_rep = None
    else:
        plot_mode = "best_plus_stuck"
        single_rep = None
        non_escaped = [r for r in rep_summaries if not r["escaped"]]
        stuck_rep = min(non_escaped, key=lambda r: abs(r["final_clean_eval"] - clean_ref_final))

    def pack(rep_obj):
        if rep_obj is None:
            return None
        return {
            "rep": int(rep_obj["rep"]),
            "eval_hist": rep_obj["eval_hist"].tolist(),
            "final_clean_eval": float(rep_obj["final_clean_eval"]),
            "min_clean_eval_over_traj": float(rep_obj["min_clean_eval_over_traj"]),
            "escaped": bool(rep_obj["escaped"]),
        }

    return {
        "plot_mode": plot_mode,
        "single_rep": pack(single_rep),
        "best_rep": pack(best_rep),
        "stuck_rep": pack(stuck_rep),
        "escape_count": escape_count,
        "num_repeats": num_repeats,
    }


# =============================================================================
# Shot-noise experiments
# =============================================================================
def run_shot_experiments(init_params_same, steps, clean_ref_final, escape_threshold):
    results = []

    for shots in SHOT_LIST:
        print(f"\n[Shot noise | opt={SHOT_OPTIMIZER}] shots = {shots}")
        reps = []

        for rep in range(SHOT_REPEATS):
            device_seed = SHOT_DEVICE_BASE_SEED + 10000 * shots + rep
            shot_cost = make_shot_cost(shots, device_seed)

            t0 = time.time()
            res = optimize_fixed_steps(
                cost_fn=shot_cost,
                params0=init_params_same,
                steps=steps,
                optimizer_type=SHOT_OPTIMIZER,
                lr_gd=LR_SHOT_GD,
                lr_adam=LR_SHOT_ADAM,
                spsa_alpha=SHOT_SPSA_ALPHA,
                spsa_gamma=SHOT_SPSA_GAMMA,
                spsa_c=SHOT_SPSA_C,
                spsa_A=SHOT_SPSA_A,
                spsa_a=SHOT_SPSA_a,
                eval_fn=CLEAN_COST,
            )

            item = {
                "rep": int(rep),
                "device_seed": int(device_seed),
                "train_hist": res["train_hist"].tolist(),
                "eval_hist": res["eval_hist"].tolist(),
                "final_shot_obj": float(res["final_train"]),
                "final_clean_eval": float(res["final_eval"]),
                "time_sec": float(time.time() - t0),
            }
            reps.append(item)

            print(
                f"  rep={rep:02d} | final shot obj={item['final_shot_obj']:.8f} | "
                f"final clean eval={item['final_clean_eval']:.8f} | "
                f"time={item['time_sec']:.1f}s"
            )

        finals = np.array([r["final_clean_eval"] for r in reps], dtype=float)

        shot_summary = summarize_shot_runs_for_plot(
            repeats=reps,
            clean_ref_final=clean_ref_final,
            escape_threshold=escape_threshold,
        )

        results.append({
            "shots": int(shots),
            "optimizer_type": SHOT_OPTIMIZER,
            "lr_gd": float(LR_SHOT_GD),
            "lr_adam": float(LR_SHOT_ADAM),
            "spsa_alpha": float(SHOT_SPSA_ALPHA),
            "spsa_gamma": float(SHOT_SPSA_GAMMA),
            "spsa_c": float(SHOT_SPSA_C),
            "spsa_A": None if SHOT_SPSA_A is None else float(SHOT_SPSA_A),
            "spsa_a": None if SHOT_SPSA_a is None else float(SHOT_SPSA_a),
            "repeats": reps,
            "mean_final_clean_eval": float(finals.mean()),
            "std_final_clean_eval": float(finals.std()),
            "escape_threshold": float(escape_threshold),
            "escape_count": int(shot_summary["escape_count"]),
            "num_repeats": int(shot_summary["num_repeats"]),
            "plot_mode": shot_summary["plot_mode"],
            "single_rep": shot_summary["single_rep"],
            "best_rep": shot_summary["best_rep"],
            "stuck_rep": shot_summary["stuck_rep"],
        })

    return results


# =============================================================================
# Plot
# =============================================================================
def plot_results(init_seed, clean_run, pauli_run, shot_runs, prefix):
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))

    # Panel 1
    ax = axes[0]

    x_clean = np.arange(len(clean_run["eval_hist"]))
    ax.plot(
        x_clean,
        clean_run["eval_hist"],
        lw=2.0,
        label=f"Clean ({clean_run['optimizer_type']})"
    )

    ax.plot(
        pauli_run["history_steps"],
        pauli_run["history"],
        "o-",
        lw=1.8,
        ms=4,
        label=f"Adaptive Pauli ({pauli_run['optimizer_type']}, clean-eval ckpt)"
    )

    for item in shot_runs:
        if item["plot_mode"] == "best_plus_stuck":
            best_hist = np.array(item["best_rep"]["eval_hist"], dtype=float)
            stuck_hist = np.array(item["stuck_rep"]["eval_hist"], dtype=float)

            ax.plot(
                np.arange(len(best_hist)),
                best_hist,
                lw=1.6,
                label=f"Shot={item['shots']} best-min (rep {item['best_rep']['rep']})"
            )
            ax.plot(
                np.arange(len(stuck_hist)),
                stuck_hist,
                lw=1.2,
                ls="--",
                label=f"Shot={item['shots']} stuck (rep {item['stuck_rep']['rep']})"
            )
        else:
            single_hist = np.array(item["single_rep"]["eval_hist"], dtype=float)

            if item["plot_mode"] == "single_stuck_only":
                label = (
                    f"Shot={item['shots']} rep-only "
                    f"(0/{item['num_repeats']} escaped, rep {item['single_rep']['rep']})"
                )
            else:
                label = (
                    f"Shot={item['shots']} best-only "
                    f"({item['num_repeats']}/{item['num_repeats']} escaped, rep {item['single_rep']['rep']})"
                )

            ax.plot(
                np.arange(len(single_hist)),
                single_hist,
                lw=1.5,
                label=label
            )

    stage_end_steps = np.cumsum(pauli_run["stage_steps"])
    for end_step in stage_end_steps[:-1]:
        ax.axvline(end_step, color="gray", ls=":", lw=0.8, alpha=0.5)

    ax.axhline(GLOBAL_MIN_ENERGY, color="red", ls="--", lw=1.2, label="Global min")
    ax.set_xlabel("Optimizer step")
    ax.set_ylabel("Clean exact energy")
    ax.set_title(f"Same init seed={init_seed}")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

    # Panel 2
    ax = axes[1]
    x = np.arange(len(pauli_run["noise_schedule"]))
    colors = ["green" if c else "red" for c in pauli_run["stage_converged"]]
    bars = ax.bar(x, pauli_run["stage_steps"], color=colors, alpha=0.75,
                  edgecolor="black", lw=0.5)

    ax.set_xticks(x)
    ax.set_xticklabels([f"p={p}" for p in pauli_run["noise_schedule"]],
                       rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Steps used")
    ax.set_title("Adaptive Pauli stage steps\n(green=converged, red=max_steps hit)")

    for bar, n in zip(bars, pauli_run["stage_steps"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(n), ha="center", va="bottom", fontsize=7)

    ax.grid(True, alpha=0.3, axis="y")

    # Panel 3
    ax = axes[2]
    labels = [
        f"clean\n({clean_run['optimizer_type']})",
        f"pauli\n({pauli_run['optimizer_type']})",
    ]
    values = [
        clean_run["final_eval"],
        pauli_run["final_clean_eval"],
    ]

    for item in shot_runs:
        if item["plot_mode"] == "best_plus_stuck":
            labels.append(f"shot={item['shots']}\nbest-min")
            values.append(item["best_rep"]["min_clean_eval_over_traj"])

            labels.append(f"shot={item['shots']}\nstuck-final")
            values.append(item["stuck_rep"]["final_clean_eval"])
        else:
            labels.append(f"shot={item['shots']}\nrep-only")
            values.append(item["single_rep"]["min_clean_eval_over_traj"])

    xpos = np.arange(len(labels))
    ax.bar(xpos, values, alpha=0.8, edgecolor="black", lw=0.6)
    ax.axhline(GLOBAL_MIN_ENERGY, color="red", ls="--", lw=1.2, label="Global min")
    ax.set_xticks(xpos)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_ylabel("Energy")
    ax.set_title("Reachability summary")
    ax.grid(True, alpha=0.3, axis="y")
    ax.legend(fontsize=8)

    plt.tight_layout()
    png_name = f"{prefix}_comparison.png"
    plt.savefig(png_name, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"Plot saved -> {png_name}")


# =============================================================================
# Main
# =============================================================================
def run_all():
    print("CWD =", os.getcwd())
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    prefix = f"{SAVE_PREFIX}_{ts}"

    # 1) init seed
    if RUN_SEED_SEARCH:
        search_result = find_saddle_like_seed()
        target = search_result["selected"]
        problematic_sorted = search_result["problematic_sorted"]
    else:
        if INIT_SEED is None:
            raise ValueError("RUN_SEED_SEARCH=False 인데 INIT_SEED가 None입니다.")
        target = diagnose_seed(int(INIT_SEED))
        problematic_sorted = [target] if is_problematic_seed(target) else []
        if problematic_sorted:
            problematic_sorted = sort_problematic_seeds(problematic_sorted)
            print_problematic_seeds(problematic_sorted)

    init_seed = target["seed"]
    init_params_same = target["init_params"]
    target_kind = target["kind"]
    target_clean = target["clean_result"]
    target_hess = target["hessian_info"]
    target_gap = target["gap"]
    target_paper_trap = target["paper_trap_info"]
    target_trap_score = target["trap_score_info"]

    np.save(f"{prefix}_init_params_seed{init_seed}.npy", np.array(init_params_same))

    print("\n" + "=" * 90)
    print(f"Using init seed       : {init_seed}")
    print(f"Seed diagnosis        : {target_kind}")
    print(f"Clean endpoint energy : {target_clean['final_eval']:.8f}")
    print(f"Suboptimal gap        : {target_gap:.8f}")
    print(
        f"Hessian/grad info     : ||grad|| = {target_hess['grad_norm']:.3e}, "
        f"lambda_min = {target_hess['lambda_min']:.3e}, "
        f"lambda_max = {target_hess['lambda_max']:.3e}"
    )
    print(
        f"Paper-trap info       : trapped = {target_paper_trap['trapped']} | "
        f"tail_std = {target_paper_trap['tail_std']:.3e} | "
        f"tail_drift = {target_paper_trap['tail_drift']:.3e}"
    )
    print(
        f"Trap-score info       : score = {target_trap_score['score']:.3e} | "
        f"tail_std = {target_trap_score['flatness_std']:.3e} | "
        f"tail_drift = {target_trap_score['flatness_drift']:.3e}"
    )
    print("=" * 90)

    # 2) adaptive Pauli first
    print(f"\n[Adaptive Pauli annealing | optimizer={PAULI_OPTIMIZER}]")
    t0 = time.time()
    pauli_run = run_pauli_annealing(
        init_params_same=init_params_same,
        noise_schedule=PAULI_SCHEDULE,
        optimizer_type=PAULI_OPTIMIZER,
        lr_gd=LR_PAULI_GD,
        lr_adam=LR_PAULI_ADAM,
        spsa_alpha=PAULI_SPSA_ALPHA,
        spsa_gamma=PAULI_SPSA_GAMMA,
        spsa_c=PAULI_SPSA_C,
        spsa_A=PAULI_SPSA_A,
        spsa_a=PAULI_SPSA_a,
        min_steps=PAULI_MIN_STEPS,
        max_steps=PAULI_MAX_STEPS,
        check_every=PAULI_CHECK_EVERY,
        window_steps=PAULI_WINDOW,
        std_tol=PAULI_STD_TOL,
        rate_tol=PAULI_RATE_TOL,
    )
    print(f"\nAdaptive Pauli done: final clean eval = {pauli_run['final_clean_eval']:.8f}")
    print(f"optimizer = {pauli_run['optimizer_type']}")
    print(f"total_steps = {pauli_run['total_steps']}")
    print(f"clean-landscape ||grad|| = {pauli_run['final_grad_norm_clean_landscape']:.3e}")
    print(f"clean-landscape lambda_min = {pauli_run['final_lambda_min_clean_landscape']:.3e}")
    print(f"time = {time.time() - t0:.1f}s")

    np.save(f"{prefix}_pauli_hist_ckpt.npy", pauli_run["history"])
    np.save(f"{prefix}_pauli_hist_steps.npy", pauli_run["history_steps"])
    np.save(f"{prefix}_pauli_final_params.npy", np.array(pauli_run["final_params"]))

    compare_steps = pauli_run["total_steps"]

    # 3) clean exact baseline with same total optimizer-step budget
    print(f"\n[Clean exact baseline | optimizer={CLEAN_OPTIMIZER}]")
    t0 = time.time()
    clean_run = optimize_fixed_steps(
        cost_fn=CLEAN_COST,
        params0=init_params_same,
        steps=compare_steps,
        optimizer_type=CLEAN_OPTIMIZER,
        lr_gd=LR_CLEAN_GD,
        lr_adam=LR_CLEAN_ADAM,
        spsa_alpha=CLEAN_SPSA_ALPHA,
        spsa_gamma=CLEAN_SPSA_GAMMA,
        spsa_c=CLEAN_SPSA_C,
        spsa_A=CLEAN_SPSA_A,
        spsa_a=CLEAN_SPSA_a,
        eval_fn=CLEAN_COST,
    )
    clean_info = clean_hessian_info(clean_run["final_params"])
    print(f"Clean final energy     = {clean_run['final_eval']:.8f}")
    print(f"Clean final ||grad||   = {clean_info['grad_norm']:.3e}")
    print(f"Clean final lambda_min = {clean_info['lambda_min']:.3e}")
    print(f"time = {time.time() - t0:.1f}s")

    np.save(f"{prefix}_clean_eval_hist.npy", clean_run["eval_hist"])
    np.save(f"{prefix}_clean_final_params.npy", np.array(clean_run["final_params"]))

    # shot-noise escape threshold
    if SHOT_ESCAPE_THRESHOLD is None:
        escape_threshold = float(clean_run["final_eval"] - SHOT_ESCAPE_DELTA)
    else:
        escape_threshold = float(SHOT_ESCAPE_THRESHOLD)

    print(f"\nShot escape threshold = {escape_threshold:.6f}")

    # 4) shot-noise runs with same total optimizer-step budget
    print(f"\n[Shot-noise runs | optimizer={SHOT_OPTIMIZER}]")
    shot_runs = run_shot_experiments(
        init_params_same=init_params_same,
        steps=compare_steps,
        clean_ref_final=float(clean_run["final_eval"]),
        escape_threshold=escape_threshold,
    )

    for item in shot_runs:
        shots = item["shots"]

        if item["single_rep"] is not None:
            np.save(
                f"{prefix}_shot_{shots}_single_rep_eval_hist.npy",
                np.array(item["single_rep"]["eval_hist"], dtype=float)
            )

        if item["best_rep"] is not None:
            np.save(
                f"{prefix}_shot_{shots}_best_rep_eval_hist.npy",
                np.array(item["best_rep"]["eval_hist"], dtype=float)
            )

        if item["stuck_rep"] is not None:
            np.save(
                f"{prefix}_shot_{shots}_stuck_rep_eval_hist.npy",
                np.array(item["stuck_rep"]["eval_hist"], dtype=float)
            )

    # 5) save summary
    summary = {
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "ranges": RANGES,
        "global_min_energy": GLOBAL_MIN_ENERGY,
        "run_seed_search": bool(RUN_SEED_SEARCH),
        "init_seed": int(init_seed),
        "num_problematic_seeds": int(len(problematic_sorted)),
        "problematic_seed_list_sorted_by_actual_selection_rule": [int(x["seed"]) for x in problematic_sorted],
        "seed_diagnosis": {
            "kind": target_kind,
            "clean_endpoint_energy": float(target_clean["final_eval"]),
            "suboptimal_gap": float(target_gap),
            "grad_norm": float(target_hess["grad_norm"]),
            "lambda_min": float(target_hess["lambda_min"]),
            "lambda_max": float(target_hess["lambda_max"]),
            "paper_trapped": bool(target_paper_trap["trapped"]),
            "paper_tail_std": float(target_paper_trap["tail_std"]),
            "paper_tail_drift": float(target_paper_trap["tail_drift"]),
            "paper_only_score": float(paper_only_score(target)),
            "trap_score": float(target_trap_score["score"]),
            "trap_tail_std": float(target_trap_score["flatness_std"]),
            "trap_tail_drift": float(target_trap_score["flatness_drift"]),
        },
        "clean_optimizer": CLEAN_OPTIMIZER,
        "shot_optimizer": SHOT_OPTIMIZER,
        "pauli_optimizer": PAULI_OPTIMIZER,
        "clean_spsa": {
            "alpha": float(CLEAN_SPSA_ALPHA),
            "gamma": float(CLEAN_SPSA_GAMMA),
            "c": float(CLEAN_SPSA_C),
            "A": None if CLEAN_SPSA_A is None else float(CLEAN_SPSA_A),
            "a": None if CLEAN_SPSA_a is None else float(CLEAN_SPSA_a),
        },
        "shot_spsa": {
            "alpha": float(SHOT_SPSA_ALPHA),
            "gamma": float(SHOT_SPSA_GAMMA),
            "c": float(SHOT_SPSA_C),
            "A": None if SHOT_SPSA_A is None else float(SHOT_SPSA_A),
            "a": None if SHOT_SPSA_a is None else float(SHOT_SPSA_a),
        },
        "pauli_spsa": {
            "alpha": float(PAULI_SPSA_ALPHA),
            "gamma": float(PAULI_SPSA_GAMMA),
            "c": float(PAULI_SPSA_C),
            "A": None if PAULI_SPSA_A is None else float(PAULI_SPSA_A),
            "a": None if PAULI_SPSA_a is None else float(PAULI_SPSA_a),
        },
        "lr_clean_gd": LR_CLEAN_GD,
        "lr_clean_adam": LR_CLEAN_ADAM,
        "lr_shot_gd": LR_SHOT_GD,
        "lr_shot_adam": LR_SHOT_ADAM,
        "lr_pauli_gd": LR_PAULI_GD,
        "lr_pauli_adam": LR_PAULI_ADAM,
        "search_steps": SEARCH_STEPS,
        "grad_tol": GRAD_TOL,
        "pauli_schedule": PAULI_SCHEDULE,
        "pauli_min_steps": PAULI_MIN_STEPS,
        "pauli_max_steps": PAULI_MAX_STEPS,
        "pauli_check_every": PAULI_CHECK_EVERY,
        "pauli_window": PAULI_WINDOW,
        "compare_steps": int(compare_steps),
        "shot_escape_threshold": float(escape_threshold),
        "clean_reference": {
            "optimizer_type": clean_run["optimizer_type"],
            "final_clean_eval": float(clean_run["final_eval"]),
            "final_grad_norm": float(clean_info["grad_norm"]),
            "lambda_min": float(clean_info["lambda_min"]),
        },
        "adaptive_pauli": {
            "optimizer_type": pauli_run["optimizer_type"],
            "final_clean_eval": float(pauli_run["final_clean_eval"]),
            "total_steps": int(pauli_run["total_steps"]),
            "stage_steps": pauli_run["stage_steps"],
            "stage_stds": pauli_run["stage_stds"],
            "stage_converged": pauli_run["stage_converged"],
            "final_grad_norm_clean_landscape": float(pauli_run["final_grad_norm_clean_landscape"]),
            "final_lambda_min_clean_landscape": float(pauli_run["final_lambda_min_clean_landscape"]),
        },
        "shot_runs": shot_runs,
    }

    json_name = f"{prefix}_summary.json"
    with open(json_name, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print(f"\nSummary saved -> {json_name}")

    # 6) plot
    plot_results(
        init_seed=init_seed,
        clean_run=clean_run,
        pauli_run=pauli_run,
        shot_runs=shot_runs,
        prefix=prefix,
    )

    # 7) console summary
    print("\n" + "=" * 180)
    print(
        f"{'mode':>24} | {'optimizer':>10} | {'setting':>14} | "
        f"{'final/mean':>18} | {'steps':>8} | {'escape count':>14} | "
        f"{'best min reached':>16} | {'stuck rep final':>16}"
    )
    print(
        f"{'clean exact':>24} | {clean_run['optimizer_type']:>10} | {'-':>14} | "
        f"{clean_run['final_eval']:>18.8f} | {compare_steps:>8} | {'-':>14} | "
        f"{'-':>16} | {'-':>16}"
    )
    print(
        f"{'adaptive pauli':>24} | {pauli_run['optimizer_type']:>10} | {'schedule':>14} | "
        f"{pauli_run['final_clean_eval']:>18.8f} | {pauli_run['total_steps']:>8} | {'-':>14} | "
        f"{'-':>16} | {'-':>16}"
    )

    for item in shot_runs:
        escape_str = f"{item['escape_count']}/{item['num_repeats']}"
        if item["plot_mode"] == "best_plus_stuck":
            best_min_str = f"{item['best_rep']['min_clean_eval_over_traj']:.8f}"
            stuck_final_str = f"{item['stuck_rep']['final_clean_eval']:.8f}"
        else:
            best_min_str = f"{item['single_rep']['min_clean_eval_over_traj']:.8f}"
            stuck_final_str = "-"

        print(
            f"{'shot noise':>24} | {item['optimizer_type']:>10} | {str(item['shots']):>14} | "
            f"{item['mean_final_clean_eval']:>18.8f} | {compare_steps:>8} | "
            f"{escape_str:>14} | {best_min_str:>16} | {stuck_final_str:>16}"
        )

    print("=" * 180)

    return summary


# =============================================================================
# Execute
# =============================================================================
results = run_all()